In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_absolute_error, accuracy_score, confusion_matrix
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.preprocessing import RobustScaler, MinMaxScaler, LabelEncoder

In [2]:
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DATA_FILENAME = 'final_data_outliers_clean_Iso.csv'
DATA_PATH_CANDIDATES = (
    Path('software_host') / 'data' / DATA_FILENAME,
    Path('..') / 'data' / DATA_FILENAME,
    Path('data') / DATA_FILENAME,
)
DATA_PATH = next((p for p in DATA_PATH_CANDIDATES if p.is_file()), None)
if DATA_PATH is None:
    searched = '\n'.join(str(p.resolve()) for p in DATA_PATH_CANDIDATES)
    raise FileNotFoundError(f'Could not find {DATA_FILENAME}. Searched:\n{searched}')

CLF_FREQ_INDICES = [0, 1, 2, 9, 10, 12, 13, 14]   #Same freq with classification scaling_classification.ipynb
BATCH_SIZE = 64
EPOCHS = 150
EARLY_STOP_PATIENCE = 12
OUTPUT_XLSX = 'validation_predictions_cascade.xlsx'

TARGETS_BY_LABEL = {
    'Resistor': ['Value_R'],
    'Capacitor': ['Value_C'],
    'Series_RC': ['Value_R', 'Value_C'],
    'Parallel_RC': ['Value_R', 'Value_C'],
}

In [3]:
class SpectraDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        return self.X[i], self.y[i]


class CNNRegressor(nn.Module):

    def __init__(self, num_outputs):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv1d(2, 8, kernel_size=3, padding=1),
            nn.BatchNorm1d(8), nn.ReLU(), nn.Dropout1d(0.1), nn.MaxPool1d(2),
            nn.Conv1d(8, 16, kernel_size=3, padding=1),
            nn.BatchNorm1d(16), nn.ReLU(), nn.Dropout1d(0.1),
            nn.Flatten(),
        )
        self.head = nn.Sequential(
            nn.Linear(16 * 7, 32), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, num_outputs),
        )

    def forward(self, x):
        return self.head(self.backbone(x))


def make_X(df_sub, scaler_mag, scaler_phase, mag_cols, phase_cols):
    Xm = scaler_mag.transform(np.log10(df_sub[mag_cols]))
    Xp = scaler_phase.transform(df_sub[phase_cols])
    return np.concatenate([Xm[:, None, :], Xp[:, None, :]], axis=1)


def train_regressor(label, train_sub, val_sub, mag_cols, phase_cols, target_cols, device):
    """Train CNNRegressor independently for 1 Label. Returns model + scalers
    + predictions (real values) on val_sub of the same Label (oracle-routing)."""
    scaler_mag = RobustScaler().fit(np.log10(train_sub[mag_cols]))
    scaler_phase = RobustScaler().fit(train_sub[phase_cols])
    scaler_y = RobustScaler().fit(np.log10(train_sub[target_cols]))

    X_tr = make_X(train_sub, scaler_mag, scaler_phase, mag_cols, phase_cols)
    X_va = make_X(val_sub, scaler_mag, scaler_phase, mag_cols, phase_cols)
    y_tr = scaler_y.transform(np.log10(train_sub[target_cols]))
    y_va = scaler_y.transform(np.log10(val_sub[target_cols])) if len(val_sub) else np.zeros((0, len(target_cols)))

    train_loader = DataLoader(SpectraDataset(X_tr, y_tr), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(SpectraDataset(X_va, y_va), batch_size=BATCH_SIZE, shuffle=False)

    model = CNNRegressor(len(target_cols)).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    criterion = nn.HuberLoss()
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

    checkpoint = Path(f'best_model_{label}.pth')
    best_val = float('inf')
    no_improve = 0

    for epoch in range(EPOCHS):
        model.train()
        tr_loss = 0.0 
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X), y)
            loss.backward()
            optimizer.step()
            tr_loss += loss.item() * X.size(0) 
            
        tr_loss /= max(len(train_loader.dataset), 1)

        model.eval()
        va_loss = 0.0
        with torch.no_grad():
            for X, y in val_loader:
                X, y = X.to(device), y.to(device)
                va_loss += criterion(model(X), y).item() * X.size(0)
        va_loss /= max(len(val_loader.dataset), 1)
        scheduler.step(va_loss)

        mae_str = ""
        if len(val_sub) > 0:
            with torch.no_grad():
                preds_scaled = model(torch.tensor(X_va, dtype=torch.float32).to(device)).cpu().numpy()
            
            preds_real = 10 ** scaler_y.inverse_transform(preds_scaled)
            y_real_val = val_sub[target_cols].values

            mae_strs = []
            if 'Value_R' in target_cols:
                idx_r = target_cols.index('Value_R')
                mae_r = mean_absolute_error(y_real_val[:, idx_r], preds_real[:, idx_r])
                mae_strs.append(f"MAE_R: {mae_r:.1f} Ohm")
            if 'Value_C' in target_cols:
                idx_c = target_cols.index('Value_C')
                mae_c = mean_absolute_error(y_real_val[:, idx_c], preds_real[:, idx_c])
                mae_strs.append(f"MAE_C: {mae_c:.3f} C")
                
            mae_str = " || " + " | ".join(mae_strs)

        print(f"Epoch {epoch+1:03d} | Train L: {tr_loss:.4f} | Val L: {va_loss:.4f}{mae_str}")

        if va_loss < best_val:
            best_val = va_loss
            no_improve = 0
            torch.save(model.state_dict(), checkpoint)
        else:
            no_improve += 1
            if no_improve >= EARLY_STOP_PATIENCE:
                print(f"Early stopping at epoch {epoch+1}")
                break

    with torch.no_grad():
        preds_scaled = model(torch.tensor(X_va, dtype=torch.float32).to(device)).cpu().numpy()
    preds_real = 10 ** scaler_y.inverse_transform(preds_scaled) if len(val_sub) else preds_scaled

    print(f'[{label}] best val Huber loss: {best_val:.4f} | n_train={len(train_sub)} n_val={len(val_sub)}')
    return {
        'model': model, 'scaler_mag': scaler_mag, 'scaler_phase': scaler_phase,
        'scaler_y': scaler_y, 'target_cols': target_cols, 'oracle_preds': preds_real,
    }

In [4]:
def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    df = pd.read_csv(DATA_PATH)
    df = df[(df['Label'] != 'nothing') & ((df['Value_R'] > 0) | (df['Value_C'] > 0))].copy()
    df['Group_ID'] = df['Label'] + '_' + df['Value_R'].astype(str) + '_' + df['Value_C'].astype(str)

    # Group split -> REGRESSION stage only. Whole (Label, R, C) groups are held out so the
    # parameter regressors are tested on component values never seen in training.
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
    train_idx, val_idx = next(splitter.split(df, groups=df['Group_ID']))
    train_df = df.iloc[train_idx].copy()
    val_df = df.iloc[val_idx].copy().reset_index(drop=True)

    mag_cols = [f'Magnitude_{i}' for i in range(15)]
    phase_cols = [f'Phase_{i}' for i in range(15)]
    clf_cols = []
    for idx in CLF_FREQ_INDICES:
        clf_cols += [f'Magnitude_{idx}', f'Phase_{idx}']

    # ---------- Classifier (scaling_classification.ipynb) ----------
    # Stratified (non-group) split -> CLASSIFICATION stage only. Topology does not depend on
    # the exact R/C value, so a plain stratified split is used here (matching
    # scaling_classification.ipynb); the group split above is kept for the regression stage.
    clf_train_df, clf_val_df = train_test_split(
        df, test_size=0.3, stratify=df['Label'], random_state=SEED
    )

    le = LabelEncoder().fit(clf_train_df['Label'])
    clf_scaler = MinMaxScaler().fit(clf_train_df[clf_cols])
    clf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=SEED)
    clf.fit(clf_scaler.transform(clf_train_df[clf_cols]), le.transform(clf_train_df['Label']))

    clf_val_pred = le.inverse_transform(clf.predict(clf_scaler.transform(clf_val_df[clf_cols])))
    clf_acc = accuracy_score(clf_val_df['Label'], clf_val_pred)
    print(f'Classifier (stratified, non-group) accuracy: {clf_acc:.4f}\n')

    # Predicted labels on the regression val set (group-split) -> used for end-to-end routing.
    val_pred_label = le.inverse_transform(clf.predict(clf_scaler.transform(val_df[clf_cols])))

    # ---------- Regressor based on Label ----------
    regressors = {}
    for label, target_cols in TARGETS_BY_LABEL.items():
        train_sub = train_df[train_df['Label'] == label]
        val_sub = val_df[val_df['Label'] == label]
        regressors[label] = train_regressor(
            label, train_sub, val_sub, mag_cols, phase_cols, target_cols, device
        )

    # ---------- Oracle routing (Label that generated) ----------
    oracle_rows = []
    for label, info in regressors.items():
        val_sub = val_df[val_df['Label'] == label].reset_index(drop=True)
        preds = info['oracle_preds']
        pred_R = preds[:, info['target_cols'].index('Value_R')] if 'Value_R' in info['target_cols'] else np.zeros(len(val_sub))
        pred_C = preds[:, info['target_cols'].index('Value_C')] if 'Value_C' in info['target_cols'] else np.zeros(len(val_sub))
        oracle_rows.append(pd.DataFrame({
            'Label': val_sub['Label'], 'Group_ID': val_sub['Group_ID'],
            'Actual_R': val_sub['Value_R'], 'Predicted_R': pred_R,
            'Actual_C': val_sub['Value_C'], 'Predicted_C': pred_C,
        }))
    oracle_df = pd.concat(oracle_rows, ignore_index=True)
    oracle_df['AbsError_R'] = np.abs(oracle_df['Predicted_R'] - oracle_df['Actual_R'])
    oracle_df['AbsError_C'] = np.abs(oracle_df['Predicted_C'] - oracle_df['Actual_C'])

    # ---------- End-to-end (Label predicted by classifier) ----------
    e2e_pred_R = np.zeros(len(val_df))
    e2e_pred_C = np.zeros(len(val_df))
    for label, info in regressors.items():
        mask = val_pred_label == label
        if not mask.any():
            continue
        sub = val_df[mask]
        X_sub = make_X(sub, info['scaler_mag'], info['scaler_phase'], mag_cols, phase_cols)
        with torch.no_grad():
            preds_scaled = info['model'](torch.tensor(X_sub, dtype=torch.float32).to(device)).cpu().numpy()
        preds_real = 10 ** info['scaler_y'].inverse_transform(preds_scaled)
        if 'Value_R' in info['target_cols']:
            e2e_pred_R[mask] = preds_real[:, info['target_cols'].index('Value_R')]
        if 'Value_C' in info['target_cols']:
            e2e_pred_C[mask] = preds_real[:, info['target_cols'].index('Value_C')]

    e2e_df = pd.DataFrame({
        'Label': val_df['Label'], 'Predicted_Label': val_pred_label, 'Group_ID': val_df['Group_ID'],
        'Actual_R': val_df['Value_R'], 'Predicted_R': e2e_pred_R,
        'Actual_C': val_df['Value_C'], 'Predicted_C': e2e_pred_C,
    })
    e2e_df['AbsError_R'] = np.abs(e2e_df['Predicted_R'] - e2e_df['Actual_R'])
    e2e_df['AbsError_C'] = np.abs(e2e_df['Predicted_C'] - e2e_df['Actual_C'])
    e2e_df['Misclassified'] = e2e_df['Label'] != e2e_df['Predicted_Label']

    def summarize(g):
        return pd.Series({'n_samples': len(g), 'MAE_R': g['AbsError_R'].mean(), 'MAE_C': g['AbsError_C'].mean()})

    oracle_by_label = oracle_df.groupby('Label').apply(summarize, include_groups=False).reset_index()
    oracle_by_group = oracle_df.groupby('Group_ID').apply(summarize, include_groups=False).reset_index()
    e2e_by_label = e2e_df.groupby('Label').apply(summarize, include_groups=False).reset_index()
    e2e_by_group = e2e_df.groupby('Group_ID').apply(summarize, include_groups=False).reset_index()

    labels_sorted = sorted(df['Label'].unique())
    cm = confusion_matrix(clf_val_df['Label'], clf_val_pred, labels=labels_sorted)
    cm_df = pd.DataFrame(cm, index=labels_sorted, columns=labels_sorted)

    with pd.ExcelWriter(OUTPUT_XLSX, engine='openpyxl') as writer:
        pd.DataFrame([{'Accuracy': clf_acc, 'n_val': len(clf_val_df)}]).to_excel(writer, sheet_name='Classifier', index=False)
        cm_df.to_excel(writer, sheet_name='Classifier', startrow=3)
        oracle_by_label.to_excel(writer, sheet_name='Oracle_by_Label', index=False)
        oracle_by_group.to_excel(writer, sheet_name='Oracle_by_Group', index=False)
        e2e_by_label.to_excel(writer, sheet_name='EndToEnd_by_Label', index=False)
        e2e_by_group.to_excel(writer, sheet_name='EndToEnd_by_Group', index=False)
        oracle_df.to_excel(writer, sheet_name='Oracle_Predictions', index=False)
        e2e_df.to_excel(writer, sheet_name='EndToEnd_Predictions', index=False)

    print(f'\nSaved results to {OUTPUT_XLSX}')
    print(f'\nClassifier accuracy (stratified, non-group): {clf_acc:.4f}')
    print('\nConfusion matrix:')
    print(cm_df.to_string())
    print('\n=== Oracle routing (regression quality, if classifier is perfect) ===')
    print(oracle_by_label.round(2).to_string(index=False))
    print('\n=== End-to-end (actual, using Label predicted by classifier) ===')
    print(e2e_by_label.round(2).to_string(index=False))
    n_misclassified = e2e_df['Misclassified'].sum()
    print(f'\nNumber of misclassified samples: {n_misclassified}/{len(e2e_df)} ({n_misclassified/len(e2e_df)*100:.1f}%)')
    return regressors, clf, clf_scaler, le, val_df, oracle_df, e2e_df


if __name__ == '__main__':
    main()

Classifier (stratified, non-group) accuracy: 0.9918

Epoch 001 | Train L: 0.1621 | Val L: 0.0268 || MAE_R: 625.1 Ohm
Epoch 002 | Train L: 0.1107 | Val L: 0.0196 || MAE_R: 540.2 Ohm
Epoch 003 | Train L: 0.0860 | Val L: 0.0058 || MAE_R: 259.1 Ohm
Epoch 004 | Train L: 0.0715 | Val L: 0.0055 || MAE_R: 246.0 Ohm
Epoch 005 | Train L: 0.0539 | Val L: 0.0086 || MAE_R: 353.2 Ohm
Epoch 006 | Train L: 0.0519 | Val L: 0.0072 || MAE_R: 268.9 Ohm
Epoch 007 | Train L: 0.0464 | Val L: 0.0075 || MAE_R: 304.4 Ohm
Epoch 008 | Train L: 0.0451 | Val L: 0.0067 || MAE_R: 308.4 Ohm
Epoch 009 | Train L: 0.0403 | Val L: 0.0065 || MAE_R: 257.6 Ohm
Epoch 010 | Train L: 0.0377 | Val L: 0.0059 || MAE_R: 277.4 Ohm
Epoch 011 | Train L: 0.0438 | Val L: 0.0045 || MAE_R: 215.6 Ohm
Epoch 012 | Train L: 0.0378 | Val L: 0.0045 || MAE_R: 194.3 Ohm
Epoch 013 | Train L: 0.0350 | Val L: 0.0042 || MAE_R: 185.0 Ohm
Epoch 014 | Train L: 0.0301 | Val L: 0.0052 || MAE_R: 273.3 Ohm
Epoch 015 | Train L: 0.0339 | Val L: 0.0073 || MAE_